# FormCoach · Model B: Gemini fine-tuning (Colab, no GPU needed)
Needs: Model A trained (`runs/finetune/preds_all.csv` from the Model A notebook), a Google Cloud project with billing and Vertex AI enabled, and a Cloud Storage bucket.

Base model: **gemini-3.5-flash** (the Gemini 2.5 models were shut down on Vertex AI on Oct 20, 2026). Check the current list of tunable models in the Vertex AI docs before starting.

> **After any restart** (Runtime → Restart, a disconnect, or changing the runtime type to GPU), run the two setup cells again before continuing. Everything on Drive is kept.

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
PROJECT = 'your-gcp-project'      # <- edit
BUCKET = 'gs://your-bucket'       # <- edit
REGION = 'us-central1'
D = '/content/drive/MyDrive/formcoach'
os.environ['D'] = D   # shell cells and ! commands read $D
print('Drive ready:', D)
%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
!gcloud config set project $PROJECT

In [ ]:
!git clone -q https://github.com/marimbel/formcoach-models.git /content/formcoach-models 2>/dev/null || git -C /content/formcoach-models pull -q
%cd /content/formcoach-models
!pip -q install -U google-genai

## 1. Build training situations from real reps + Model A predictions

In [ ]:
!python scripts/06_build_gemini_situations.py --data $D/processed --preds $D/runs/finetune/preds_all.csv --out $D/gemini/situations.csv

## 2. Human review (the important part)
Open `MyDrive/formcoach/gemini/situations.csv` in Google Sheets. For each row, copy `draft_json` into `final_json` and **edit it** (see `docs/gemini_labeling_guide.md`):
- review **all test rows** and all `conflict`, `uncertain`, `low_visibility` rows;
- rewrite at least ~40% of training cues in your own coaching words.

Download as CSV to `MyDrive/formcoach/gemini/situations_reviewed.csv`.

## 3. Export JSONL and upload

In [ ]:
%%bash
[ -n "$D" ] && [ -d /content/formcoach-models ] || { echo "Run the two setup cells at the top first (Drive + code). Colab forgets them after a restart or a runtime-type change."; exit 1; }
cd /content/formcoach-models
gsutil -m -q cp -r $D/processed/crops $BUCKET/formcoach/
python scripts/07_situations_to_jsonl.py --situations $D/gemini/situations_reviewed.csv --out $D/gemini \
    --image-gcs-prefix $BUCKET/formcoach/crops
gsutil -q cp $D/gemini/train.jsonl $D/gemini/val.jsonl $BUCKET/formcoach/gemini/

## 4. Fine-tune (takes roughly 30–90 minutes)
Rough cost: tokens × epochs × $0.01 per 1,000 tokens for Gemini 3.5 Flash (check current pricing). About 1,000 examples of ~600 tokens × 3 epochs ≈ $18.

In [ ]:
!python scripts/08_tune_gemini.py --project $PROJECT --location $REGION --base-model gemini-3.5-flash \
    --train $BUCKET/formcoach/gemini/train.jsonl --val $BUCKET/formcoach/gemini/val.jsonl --out $D/runs/gemini/tuning_job.json

## 5. Base vs tuned on the held-out test situations

In [ ]:
import json
job = json.load(open(f'{D}/runs/gemini/tuning_job.json'))
TUNED = job['endpoint']; print(TUNED)
!python scripts/09_eval_gemini.py --project $PROJECT --location $REGION --test $D/gemini/test_eval.jsonl \
    --data $D/processed --base gemini-3.5-flash --tuned "$TUNED" --out $D/runs/gemini/eval
from IPython.display import Image, display
display(Image(f'{D}/runs/gemini/eval/base_vs_tuned.png'))
print(open(f'{D}/runs/gemini/eval/metrics.json').read()[:3000])

In [ ]:
!python scripts/10_data_report.py --data $D/processed --gemini $D/gemini --out $D/DATA_REPORT.md
print('Set TUNED_MODEL =', TUNED, 'on the Cloud Run backend.')